# v67 temporal-neighbor augmentation — fold2 パイロット

**目的**: 枯渇宣言後の唯一残る合法定量レバー — 時間近傍GONGフレームを
学習時のみ増強(主催者裁定済み合法・推論は単一画像必須)。

各train観測の ±20分 以内の同サイトFITSフレームを最大2件取得し、
原本と同一annotationを流用してYOLO学習データを増量。
val には近傍を混入しない(純粋なfold2 valでprodとペア比較)。

**中断時は必ず「ランタイム→すべて実行」**。セル単独実行は未定義名で失敗する。
実行前に学習モデルと同じ分割の外部凍結SHA256を `CFG.GROUPED_FOLDS_SHA256` に設定する。
未設定・不一致・不正な分割では前処理前に停止する。ハッシュ一致だけで独立評価は保証しない。
全段階はDrive ckpt/キャッシュで再開可能。status: `v67_status_fold2.json`

**採否(事前宣言)**: fold2 val ΔPQ(v67−prod同条件) > +0.005 → 別fold確認。
|Δ|<0.002 ノイズ棄却、負 → このレバーも棄却。


In [ ]:
# [1/7] 依存・設定・Drive・status
%pip install -q "ultralytics>=8.3,<9" pycocotools astropy scikit-image

import json, math, random, re, shutil, time, urllib.request
from collections import defaultdict
from datetime import datetime, timedelta
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
from google.colab import drive
from tqdm.auto import tqdm

drive.mount("/content/drive")

class CFG:
    SEED = 3407
    N_FOLDS = 3
    GROUPED_FOLDS_SHA256 = ""  # Independently frozen training split SHA256 required
    FOLD = 2
    NEIGH_MINUTES = 20
    PER_OBS = 2
    DRIVE_ROOT = Path("/content/drive/MyDrive/solar_filament_v30")
    DATA_CACHE = DRIVE_ROOT / "competition_data"
    ARTIFACT_ROOT = DRIVE_ROOT / "artifacts"
    RUNS_ROOT = DRIVE_ROOT / "runs"
    PREPROC_CACHE = DRIVE_ROOT / "preprocessed_v30_q96"
    LOCAL_ROOT = Path("/content/solar_v67")
    PREPROC_DIR = LOCAL_ROOT / "preprocessed"          # val原本のみ(推論用)
    TN_FITS = DRIVE_ROOT / "temporal_neighbors" / "fits"
    TN_PREPROC = DRIVE_ROOT / "temporal_neighbors" / "preprocessed"
    TN_MANIFEST = DRIVE_ROOT / "temporal_neighbors" / "manifest.jsonl"
    YOLO_ROOT = DRIVE_ROOT / "yolo_datasets" / "v67_fold2"
    # --- YOLO検出器(prodと同一) ---
    YOLO_MODEL = "yolo11m-seg.pt"
    YOLO_IMGSZ = 1280
    YOLO_BATCH = 4
    YOLO_EPOCHS = 70
    YOLO_PATIENCE = 14
    YOLO_MASK_RATIO = 2
    # --- 後処理(prodと同一) ---
    SUBMISSION_CONF = 0.26
    REFINER_THRESHOLD = 0.45
    DROP_THRESHOLD = 0.90
    MIN_AREA = 20
    PREDICT_IOU = 0.55
    CROP_SIZE = 320
    # --- v67管理 ---
    BASELINE_PQ = 0.443200
    MANIFEST = DRIVE_ROOT / "artifacts" / "v67_temporal_aug_fold2_manifest.json"
    STATUS_FILE = DRIVE_ROOT / "artifacts" / "v67_status_fold2.json"

_T0 = time.time()
def write_status(stage, done=-1, total=-1, detail=""):
    payload = {"stage": stage, "done": done, "total": total, "detail": detail,
               "elapsed_min": round((time.time() - _T0) / 60, 1),
               "updated": time.strftime("%Y-%m-%d %H:%M:%S")}
    try:
        CFG.STATUS_FILE.write_text(json.dumps(payload, ensure_ascii=False))
    except Exception:
        pass

for path in [CFG.LOCAL_ROOT, CFG.PREPROC_DIR, CFG.TN_FITS, CFG.TN_PREPROC,
             CFG.YOLO_ROOT]:
    path.mkdir(parents=True, exist_ok=True)
random.seed(CFG.SEED); np.random.seed(CFG.SEED); torch.manual_seed(CFG.SEED)

required = {
    "competition data": CFG.DATA_CACHE,
    "grouped_folds.csv": CFG.ARTIFACT_ROOT / "grouped_folds.csv",
    "前処理画像キャッシュ": CFG.PREPROC_CACHE,
    f"fold{CFG.FOLD} prod検出器": CFG.RUNS_ROOT / f"yolo_fold_{CFG.FOLD}" / "weights" / "best.pt",
    f"fold{CFG.FOLD} 標準refiner": CFG.ARTIFACT_ROOT / f"mediseg_refiner_fold{CFG.FOLD}_resume.pt",
}
missing = {k: str(v) for k, v in required.items() if not v.exists()}
if missing:
    raise FileNotFoundError(f"前提artifactが見つかりません: {missing}")
assert torch.cuda.is_available(), "GPUランタイムを選択してください"
print({k: "OK" for k in required})
print({"GPU": torch.cuda.get_device_name(0)})
write_status("setup-ok")


In [ ]:
def load_frozen_folds(path, train_files, n_folds, expected_sha256):
    import csv
    import hashlib
    import io
    import re

    if type(n_folds) is not int or n_folds < 2:
        raise ValueError("n_folds must be an integer >= 2")
    if (not isinstance(train_files, list) or not train_files
            or any(not isinstance(name, str) or not name or name in (".", "..")
                   or name.strip() != name or any(c in name for c in "/\\:")
                   or any(ord(c) < 32 or ord(c) == 127 for c in name) for name in train_files)
            or len(set(train_files)) != len(train_files)):
        raise ValueError("Expected observations must be unique safe basenames")
    if not isinstance(expected_sha256, str) or not re.fullmatch(r"[0-9a-f]{64}", expected_sha256):
        raise ValueError("Provide the externally frozen grouped_folds.csv SHA256")
    payload = Path(path).read_bytes()
    actual_sha256 = hashlib.sha256(payload).hexdigest()
    if actual_sha256 != expected_sha256:
        raise ValueError("Frozen fold CSV SHA256 differs")
    reader = csv.DictReader(io.StringIO(payload.decode("utf-8-sig"), newline=""), strict=True)
    fields = reader.fieldnames
    if (not fields or len(fields) != len(set(fields))
            or not {"file_name", "fold"}.issubset(fields)):
        raise ValueError("Fold CSV needs unique headers including file_name and fold")
    expected, fold_of_file = set(train_files), {}
    try:
        for row in reader:
            if None in row or any(value is None for value in row.values()):
                raise ValueError("Malformed fold CSV row")
            name, value = row["file_name"], row["fold"]
            if name not in expected or name in fold_of_file:
                raise ValueError("Duplicate or unexpected fold observation")
            if not re.fullmatch(r"0|[1-9][0-9]*", value):
                raise ValueError("Fold must be an exact nonnegative integer")
            fold = int(value)
            if fold >= n_folds:
                raise ValueError("Fold is outside configured range")
            fold_of_file[name] = fold
    except csv.Error as error:
        raise ValueError("Malformed fold CSV") from error
    if set(fold_of_file) != expected or set(fold_of_file.values()) != set(range(n_folds)):
        raise ValueError("Fold CSV must cover every observation and every configured fold")
    return fold_of_file, actual_sha256

# [2/7] COCO + fold分割。val原本をSSDへ、近傍取得対象=train側のみ
def find_dataset_root(base):
    matches = list(Path(base).rglob("MAGFiLO_1.0_Annotations_kaggle2026_train.json"))
    if not matches:
        raise FileNotFoundError(f"annotation JSONが見つかりません: {base}")
    return matches[0].parents[2]

DATA_ROOT = find_dataset_root(CFG.DATA_CACHE)
TRAIN_JSON = next(DATA_ROOT.rglob("MAGFiLO_1.0_Annotations_kaggle2026_train.json"))
with open(TRAIN_JSON, encoding="utf-8") as handle:
    coco = json.load(handle)

images = {item["id"]: item for item in coco["images"]}
annotations_by_image = defaultdict(list)
for ann in coco["annotations"]:
    annotations_by_image[ann["image_id"]].append(ann)
file_to_ids = defaultdict(list)
for image_id, info in images.items():
    file_to_ids[info["file_name"]].append(image_id)
train_files = sorted(file_to_ids)

fold_of_file, frozen_fold_sha256 = load_frozen_folds(
    CFG.ARTIFACT_ROOT / "grouped_folds.csv", train_files, CFG.N_FOLDS, CFG.GROUPED_FOLDS_SHA256)
train_files_fold = [n for n in train_files if fold_of_file[n] != CFG.FOLD]
valid_files_fold = [n for n in train_files if fold_of_file[n] == CFG.FOLD]

# val原本をSSDへ(推論用)。train原本はYOLO dataset構築時にcacheから直接参照
for _ci, name in enumerate(tqdm(valid_files_fold, desc="val前処理画像をSSDへ")):
    local = CFG.PREPROC_DIR / name
    if not local.exists():
        shutil.copy2(CFG.PREPROC_CACHE / name, local)
    if _ci % 50 == 0:
        write_status("copy-val", _ci, len(valid_files_fold), name)
write_status("copy-done")

print({"除外train": len(train_files_fold), "検証(OOF)": len(valid_files_fold)})


In [ ]:
# [3/7] 近傍フレーム選定 + FITS ダウンロード(Driveキャッシュで再開可)
GONG_BASE = "https://gong2.nso.edu/ftp/HA/haf"
FILE_RE = re.compile(r'href="(\d{14}([A-Za-z])h\.fits\.fz)"')
NAME_RE = re.compile(r"(\d{14})([A-Za-z])h", re.IGNORECASE)

def fetch_url(url, binary=False):
    try:
        with urllib.request.urlopen(url, timeout=30) as r:
            data = r.read()
        return data if binary else data.decode("utf-8", errors="replace")
    except Exception as exc:
        return None

def parse_obs(file_name):
    m = NAME_RE.search(file_name)
    if not m:
        return None
    return datetime.strptime(m.group(1), "%Y%m%d%H%M%S"), m.group(2).upper()

_day_cache = {}
def list_day(day):
    key = f"{day:%Y%m%d}"
    if key not in _day_cache:
        html = fetch_url(f"{GONG_BASE}/{day:%Y%m}/{key}/")
        _day_cache[key] = [(m.group(1), m.group(2).upper())
                          for m in FILE_RE.finditer(html)] if html else []
    return _day_cache[key]

def neighbors_for(obs_time, site):
    cands = []
    for dd in (-1, 0, 1):
        day = obs_time + timedelta(days=dd)
        for fname, fsite in list_day(day):
            if fsite != site:
                continue
            fm = NAME_RE.search(fname)
            if not fm:
                continue
            ftime = datetime.strptime(fm.group(1), "%Y%m%d%H%M%S")
            diff = (ftime - obs_time).total_seconds() / 60.0
            if diff == 0 or abs(diff) > CFG.NEIGH_MINUTES:
                continue
            cands.append({"file": fname, "day": f"{day:%Y%m%d}",
                          "dt_min": round(diff, 2)})
    cands.sort(key=lambda c: abs(c["dt_min"]))
    return cands[:CFG.PER_OBS]

# manifest.jsonl: 1行1観測 {"src": name, "neighbors": [...]}
done_srcs = set()
if CFG.TN_MANIFEST.exists():
    for line in CFG.TN_MANIFEST.read_text(encoding="utf-8").splitlines():
        if line.strip():
            done_srcs.add(json.loads(line)["src"])

mf = open(CFG.TN_MANIFEST, "a", encoding="utf-8")
n_cand = 0
for i, name in enumerate(tqdm(train_files_fold, desc="近傍選定")):
    if name in done_srcs:
        continue
    parsed = parse_obs(name)
    if not parsed:
        continue
    obs_time, site = parsed
    cands = neighbors_for(obs_time, site)
    mf.write(json.dumps({"src": name, "obs_utc": obs_time.isoformat(),
                          "site": site, "neighbors": cands}) + "\n")
    mf.flush()
    n_cand += len(cands)
    if i % 25 == 0:
        write_status("select-neighbors", i, len(train_files_fold), name)
mf.close()

# ダウンロード(manifest全件、ファイル単位スキップ再開)
entries = [json.loads(l) for l in CFG.TN_MANIFEST.read_text(encoding="utf-8").splitlines() if l.strip()]
todo = [n for e in entries for n in e["neighbors"]]
n_dl = 0
for i, n in enumerate(tqdm(todo, desc="FITS DL")):
    dest = CFG.TN_FITS / n["file"]
    if dest.exists() and dest.stat().st_size > 100_000:
        continue
    url = f"{GONG_BASE}/{n['day'][:6]}/{n['day']}/{n['file']}"
    data = fetch_url(url, binary=True)
    if data and len(data) > 100_000:
        dest.write_bytes(data)
        n_dl += 1
    else:
        print(f"  取得失敗: {url}")
    if i % 20 == 0:
        write_status("download", i, len(todo), n["file"])
write_status("download-done", len(todo), len(todo))
print({"観測": len(entries), "近傍候補": len(todo), "新規DL": n_dl})


In [ ]:
# [4/7] FITS → 階調マッチ → 3ch前処理JPEG(原本と同じパイプライン)
from astropy.io import fits
from skimage.exposure import match_histograms

def find_train_dir():
    return next(DATA_ROOT.rglob("train_images"))

# 参照ヒストグラム: train原本5枚(ラベル非依存の画素分布のみ利用)
RAW_TRAIN_DIR = find_train_dir()
ref_names = train_files_fold[:5]
reference = np.vstack([
    cv2.imread(str(RAW_TRAIN_DIR / n), cv2.IMREAD_GRAYSCALE) for n in ref_names
])

def robust_uint8(array, mask=None, low=0.5, high=99.5):
    values = array[mask] if mask is not None and mask.any() else array.reshape(-1)
    lo, hi = np.percentile(values, [low, high])
    return np.clip((array - lo) * 255.0 / max(hi - lo, 1e-6), 0, 255).astype(np.uint8)

def solar_three_channel(gray):
    gray = gray.astype(np.float32)
    disk = gray > max(4.0, np.percentile(gray, 8))
    background = cv2.GaussianBlur(gray, (0, 0), sigmaX=80, sigmaY=80)
    flat_u8 = robust_uint8(gray / np.maximum(background, 1.0), disk, 1.0, 99.0)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(16, 16)).apply(flat_u8)
    raw_u8 = robust_uint8(gray, disk, 0.5, 99.5)
    dark = cv2.morphologyEx(clahe, cv2.MORPH_BLACKHAT, np.ones((31, 31), np.uint8))
    dark = robust_uint8(dark, disk, 0.0, 99.5)
    out = np.dstack([raw_u8, clahe, dark])
    out[~disk] = 0
    return out

def fits_to_preprocessed(fits_path):
    with fits.open(fits_path) as hdul:
        data = hdul[1].data.astype(np.float32) if len(hdul) > 1 else hdul[0].data.astype(np.float32)
    gray = np.clip(match_histograms(data, reference), 0, 255).astype(np.uint8)
    return solar_three_channel(gray)

fits_files = sorted(CFG.TN_FITS.glob("*.fits.fz"))
n_conv, n_fail = 0, 0
for i, fp in enumerate(tqdm(fits_files, desc="FITS→前処理JPEG")):
    out = CFG.TN_PREPROC / (fp.name.replace(".fits.fz", "") + "_tn.jpeg")
    if out.exists():
        continue
    try:
        img = fits_to_preprocessed(fp)
        if img.shape != (2048, 2048, 3):
            img = cv2.resize(img, (2048, 2048), interpolation=cv2.INTER_LINEAR)
        cv2.imwrite(str(out), img, [cv2.IMWRITE_JPEG_QUALITY, 96])
        n_conv += 1
    except Exception as exc:
        n_fail += 1
        print(f"  変換失敗: {fp.name} ({exc})")
    if i % 20 == 0:
        write_status("convert", i, len(fits_files), fp.name)
write_status("convert-done", len(fits_files), len(fits_files))
print({"変換": n_conv, "失敗": n_fail, "総近傍画像": len(list(CFG.TN_PREPROC.glob('*.jpeg')))})


In [ ]:
# [5/7] YOLO dataset構築: train=原本(全labeler)+近傍(同labeler流用)、val=原本のみ
def polygon_line(ann, width, height):
    polygons = [p for p in ann.get("segmentation", []) if len(p) >= 6]
    if not polygons:
        return None
    polygon = max(polygons, key=len)
    points = np.asarray(polygon, dtype=np.float32).reshape(-1, 2)
    points[:, 0] = np.clip(points[:, 0] / width, 0, 1)
    points[:, 1] = np.clip(points[:, 1] / height, 0, 1)
    return "0 " + " ".join(f"{v:.6f}" for v in points.reshape(-1))

def label_lines(image_id, info):
    return [polygon_line(ann, info["width"], info["height"])
            for ann in annotations_by_image[image_id] if not ann.get("iscrowd", 0)]

def write_variant(split, image_path, stem, lines):
    image_out = CFG.YOLO_ROOT / "images" / split / f"{stem}.jpeg"
    label_out = CFG.YOLO_ROOT / "labels" / split / f"{stem}.txt"
    image_out.parent.mkdir(parents=True, exist_ok=True)
    if not image_out.exists():
        shutil.copy2(image_path, image_out)
    label_out.parent.mkdir(parents=True, exist_ok=True)
    label_out.write_text("\n".join(l for l in lines if l), encoding="utf-8")

# src file_name -> 近傍前処理画像リスト の対応表
src_to_neighbors = defaultdict(list)
for e in entries:
    for n in e["neighbors"]:
        pp = CFG.TN_PREPROC / (n["file"].replace(".fits.fz", "") + "_tn.jpeg")
        if pp.exists():
            src_to_neighbors[e["src"]].append((n["file"], pp))

n_orig = n_neigh = 0
for name in tqdm(train_files_fold, desc="train側構築"):
    for image_id in file_to_ids[name]:
        info = images[image_id]
        lines = label_lines(image_id, info)
        stem = Path(name).stem + f"__labeler_{image_id}"
        write_variant("train", CFG.PREPROC_CACHE / name, stem, lines)
        n_orig += 1
        # 近傍: 同じlabelerのラベルを流用(filamentは±数分でほぼ不変)
        for fits_name, pp in src_to_neighbors.get(name, []):
            nstem = Path(pp).stem + f"__labeler_{image_id}"
            write_variant("train", pp, nstem, lines)
            n_neigh += 1

for name in tqdm(valid_files_fold, desc="val側構築"):
    image_id = file_to_ids[name][0]  # reference labeler のみ(prodと同じ)
    info = images[image_id]
    write_variant("val", CFG.PREPROC_DIR / name, Path(name).stem,
                   label_lines(image_id, info))

yaml_path = CFG.YOLO_ROOT / "data.yaml"
yaml_path.write_text(
    f"path: {CFG.YOLO_ROOT}\ntrain: images/train\nval: images/val\nnames:\n  0: filament\n",
    encoding="utf-8")
write_status("dataset-done")
print({"train原本variant": n_orig, "近傍variant": n_neigh,
       "増強倍率": round((n_orig + n_neigh) / max(n_orig, 1), 2), "yaml": str(yaml_path)})


In [ ]:
# [6/7] YOLO fold2 を近傍増強データで学習(prodと同一ハイパラ)
from ultralytics import YOLO

run_dir = CFG.RUNS_ROOT / f"v67_yolo_fold_{CFG.FOLD}"
best_path = run_dir / "weights" / "best.pt"
done_path = run_dir / "TRAINING_COMPLETE"
if done_path.exists() and best_path.exists():
    print(f"学習済みcheckpointを再利用 {best_path}")
else:
    model = YOLO(CFG.YOLO_MODEL)
    model.train(
        data=str(yaml_path), epochs=CFG.YOLO_EPOCHS, imgsz=CFG.YOLO_IMGSZ,
        batch=CFG.YOLO_BATCH, device=0, workers=4, patience=CFG.YOLO_PATIENCE,
        seed=CFG.SEED + CFG.FOLD, deterministic=True, single_cls=True,
        mask_ratio=CFG.YOLO_MASK_RATIO, overlap_mask=False, amp=True,
        cache="disk",
        optimizer="AdamW", lr0=7e-4, lrf=0.05, weight_decay=5e-4, warmup_epochs=3,
        degrees=180, translate=0.04, scale=0.25, fliplr=0.5, flipud=0.5,
        hsv_h=0.0, hsv_s=0.0, hsv_v=0.18, mosaic=0.20, close_mosaic=10,
        project=str(CFG.RUNS_ROOT), name=f"v67_yolo_fold_{CFG.FOLD}", exist_ok=True,
    )
    done_path.write_text("ok", encoding="utf-8")
write_status("train-done")
print({"v67_weight": str(best_path)})


In [ ]:
# [7/7] ペア評価: prod検出器 vs v67検出器 → refiner → ownership → PQ(fold2 val)
from pycocotools import mask as mask_utils
from ultralytics import YOLO

"""
MediSeg-AI Knowledge Port for Solar Filament Segmentation
Includes:
1. MediSeg UNet
2. MediSeg DeepLabV3Plus with ASPP (Atrous Spatial Pyramid Pooling)
3. CombinedLoss (BCE + Dice Loss)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional

# ============================================================
# 1. Combined Loss (BCE + Dice Loss)
# ============================================================
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        probs = probs.view(-1)
        targets = targets.view(-1)
        
        intersection = (probs * targets).sum()
        dice = (2.0 * intersection + self.smooth) / (probs.sum() + targets.sum() + self.smooth)
        return 1.0 - dice

class CombinedLoss(nn.Module):
    def __init__(self, bce_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        dice_loss = self.dice(logits, targets)
        return self.bce_weight * bce_loss + self.dice_weight * dice_loss

# ============================================================
# 2. MediSeg UNet Architecture
# ============================================================
class DoubleConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, mid_channels: Optional[int] = None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.double_conv(x)

class Down(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.maxpool_conv(x)

class Up(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, bilinear: bool = True):
        super().__init__()
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor:
        x1 = self.up(x1)
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2, diffY // 2, diffY - diffY // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class MediSegUNet(nn.Module):
    def __init__(self, in_channels: int = 3, out_channels: int = 1, bilinear: bool = False):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        self.down4 = Down(512, 1024 // factor)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        return self.outc(x)

# ============================================================
# 3. MediSeg DeepLabV3Plus with ASPP Module
# ============================================================
class ASPP(nn.Module):
    def __init__(self, in_channels: int, out_channels: int = 256):
        super().__init__()
        self.conv1x1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn1x1 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=6, dilation=6, bias=False)
        self.bn3x3_1 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_2 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=12, dilation=12, bias=False)
        self.bn3x3_2 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_3 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=18, dilation=18, bias=False)
        self.bn3x3_3 = nn.BatchNorm2d(out_channels)
        
        self.global_avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv_global = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn_global = nn.BatchNorm2d(out_channels)
        
        self.conv_final = nn.Conv2d(out_channels * 5, out_channels, kernel_size=1, bias=False)
        self.bn_final = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        size = x.shape[2:]
        x1 = self.relu(self.bn1x1(self.conv1x1(x)))
        x2 = self.relu(self.bn3x3_1(self.conv3x3_1(x)))
        x3 = self.relu(self.bn3x3_2(self.conv3x3_2(x)))
        x4 = self.relu(self.bn3x3_3(self.conv3x3_3(x)))
        
        x5 = self.global_avg_pool(x)
        x5 = self.relu(self.bn_global(self.conv_global(x5)))
        x5 = F.interpolate(x5, size=size, mode='bilinear', align_corners=True)
        
        x_cat = torch.cat([x1, x2, x3, x4, x5], dim=1)
        return self.relu(self.bn_final(self.conv_final(x_cat)))

class MediSegDeepLabV3Plus(nn.Module):
    def __init__(self, in_channels: int = 3, out_channels: int = 1):
        super().__init__()
        # Feature Extractor
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 512, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True)
        )
        self.aspp = ASPP(512, 256)
        self.classifier = nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, out_channels, kernel_size=1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        input_size = x.shape[2:]
        feat = self.stem(x)
        aspp_feat = self.aspp(feat)
        out = self.classifier(aspp_feat)
        return F.interpolate(out, size=input_size, mode='bilinear', align_corners=True)

# ============================================================
# 4. Edge-Guided Attention U-Net (EG-MHSA)
# ============================================================
# 2026-08-29、WINNING_STRATEGY_2026-08-29.md §1.4のPhase 2実験用に追加。
# 参考文献: "EdgeAttNet: Towards Barb-Aware Filament Segmentation"
# (arXiv:2509.02964)。MAGFiLO上でU-Net(mIoU_pairwise 0.5724)から
# EdgeAttNet(0.6451)への改善を報告しており、barb(フィラメントの微細な
# 枝構造)の認識精度向上を狙ったアーキテクチャ。本プロジェクトの
# refiner(MediSegUNet)が抱える主要な弱点(near-miss境界品質)と直接
# 対応するため、置き換え候補として追試する。
#
# 【重要な注意】著者の公開リポジトリ(github.com/dasjar/EdgeAttNet)を
# 直接確認したところ、`model/edgeattnet_model.py`(本来UNetEdgeTransformer
# クラスが定義されているべきファイル)の中身が誤って`main.py`と同一に
# なっており、実際のアーキテクチャコードはリポジトリ上に存在しない
# (2026-08-29、`gh api`でリポジトリ全ファイルを走査して確認)。学習済み
# 重みの配布も確認できなかった。**以下は論文本文のアーキテクチャ記述
# (edge prior E=σ(Conv3x3(入力画像))をbottleneckへbilinear補間し
# 1x1convで射影、Q・Kへ加算、位置エンコーディングなし)のみを根拠にした
# 独自の再実装であり、著者オリジナル実装とのコード上の一致は未検証。**
# ベースにするU-Net本体はMediSegUNetをそのまま流用しているため、
# パラメータ総数は論文記載の値(22.6M)とは一致しない。


class EdgeGuidedMHSA(nn.Module):
    """Edge-Guided Multi-Head Self-Attention (EG-MHSA)。

    U-NetのbottleneckでSelf-Attentionを行う際、入力画像から抽出した
    エッジ事前分布をQuery/Keyに加算し、境界・barb構造への感度を高める。
    Value・出力チャネル数はbottleneckの特徴マップと同一に保つ
    (MediSegUNetへdrop-inできるよう設計)。
    """

    def __init__(self, in_channels: int, bottleneck_channels: int, num_heads: int = 8):
        super().__init__()
        if bottleneck_channels % num_heads != 0:
            raise ValueError(
                f"bottleneck_channels({bottleneck_channels}) must be divisible "
                f"by num_heads({num_heads})"
            )
        # E = sigmoid(Conv3x3(入力画像)): 単一チャンネルのエッジ事前分布
        self.edge_conv = nn.Conv2d(in_channels, 1, kernel_size=3, padding=1)
        # E' = Conv1x1(interpolate(E)): bottleneckのチャンネル数へ射影
        self.edge_proj = nn.Conv2d(1, bottleneck_channels, kernel_size=1)
        self.mhsa = nn.MultiheadAttention(
            bottleneck_channels, num_heads, batch_first=True
        )
        self.norm = nn.LayerNorm(bottleneck_channels)

    def forward(self, bottleneck: torch.Tensor, original_image: torch.Tensor) -> torch.Tensor:
        b, c, h, w = bottleneck.shape
        edge_map = torch.sigmoid(self.edge_conv(original_image))
        edge_map = F.interpolate(edge_map, size=(h, w), mode="bilinear", align_corners=True)
        edge_prior = self.edge_proj(edge_map)  # (B, C, H, W)

        x_flat = bottleneck.flatten(2).transpose(1, 2)  # (B, N, C)
        edge_flat = edge_prior.flatten(2).transpose(1, 2)  # (B, N, C)
        qk = x_flat + edge_flat  # Q = K = x + E'。位置エンコーディングは使わない(論文通り)

        attn_out, _ = self.mhsa(qk, qk, x_flat)  # V = x(エッジ由来の加算なし)
        attn_out = self.norm(attn_out + x_flat)  # 残差接続 + LayerNorm(論文に明記はないが標準的なTransformerブロックとして採用)
        return attn_out.transpose(1, 2).reshape(b, c, h, w)


class MediSegEdgeAttUNet(nn.Module):
    """MediSegUNetのbottleneckにEdgeGuidedMHSAを挿入したdrop-in代替。

    入出力インターフェースはMediSegUNetと完全に同一(in_channels=3の
    crop、out_channels=1の確率マップ)。既存のrefiner学習・推論コードを
    変更せず、モデルクラスの差し替えだけで比較できるように設計している。
    """

    def __init__(
        self,
        in_channels: int = 3,
        out_channels: int = 1,
        bilinear: bool = False,
        num_heads: int = 8,
    ):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        bottleneck_channels = 1024 // factor
        self.down4 = Down(512, bottleneck_channels)
        self.edge_attn = EdgeGuidedMHSA(in_channels, bottleneck_channels, num_heads=num_heads)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x5 = self.edge_attn(x5, x)
        out = self.up1(x5, x4)
        out = self.up2(out, x3)
        out = self.up3(out, x2)
        out = self.up4(out, x1)
        return self.outc(out)


# ============================================================
# 5. Plain MHSA U-Net (matched-parameter ablation control, no edge prior)
# ============================================================
# 2026-08-31、report/main.tex Future Work item 1(matched-parameter
# ablation)のために追加。EdgeGuidedMHSAからedge_conv/edge_projを除いた
# だけの構成——Q=K=V=bottleneck特徴量の標準的なself-attentionで、
# edge priorをQ/Kへ加算する項だけがない。edge_conv/edge_projは
# 3(in_channels)*1*3*3+1 + 1*1024*1*1+1024 ≈ 2,076パラメータしかなく
# 全体(約35.2M)の0.006%程度なので、このクラスはMediSegEdgeAttUNetと
# ほぼ完全にパラメータ数が一致する("U-Net + MHSA, no positional
# encoding"に相当、EdgeAttNet論文のablationと同じ切り分け)。
# report/main.texが指摘してきた「edge priorそのものの効果か、単に
# attention容量が増えただけか」という交絡を分離するための唯一の差分。


class PlainMHSA(nn.Module):
    """edge priorなしの標準Multi-Head Self-Attention(bottleneck上)。

    EdgeGuidedMHSAと同じchannel数・head数・残差接続+LayerNormを保ちつつ、
    Q・Kへの edge_prior 加算だけを行わない(Q=K=V=bottleneck特徴量)。
    """

    def __init__(self, bottleneck_channels: int, num_heads: int = 8):
        super().__init__()
        if bottleneck_channels % num_heads != 0:
            raise ValueError(
                f"bottleneck_channels({bottleneck_channels}) must be divisible "
                f"by num_heads({num_heads})"
            )
        self.mhsa = nn.MultiheadAttention(
            bottleneck_channels, num_heads, batch_first=True
        )
        self.norm = nn.LayerNorm(bottleneck_channels)

    def forward(self, bottleneck: torch.Tensor) -> torch.Tensor:
        b, c, h, w = bottleneck.shape
        x_flat = bottleneck.flatten(2).transpose(1, 2)  # (B, N, C)
        attn_out, _ = self.mhsa(x_flat, x_flat, x_flat)  # Q=K=V=x(edge priorなし)
        attn_out = self.norm(attn_out + x_flat)
        return attn_out.transpose(1, 2).reshape(b, c, h, w)


class MediSegPlainMHSAUNet(nn.Module):
    """MediSegEdgeAttUNetからedge priorだけを除いた matched-parameter 対照群。

    入出力インターフェースはMediSegUNet/MediSegEdgeAttUNetと完全に同一。
    """

    def __init__(
        self,
        in_channels: int = 3,
        out_channels: int = 1,
        bilinear: bool = False,
        num_heads: int = 8,
    ):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        bottleneck_channels = 1024 // factor
        self.down4 = Down(512, bottleneck_channels)
        self.plain_attn = PlainMHSA(bottleneck_channels, num_heads=num_heads)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x5 = self.plain_attn(x5)
        out = self.up1(x5, x4)
        out = self.up2(out, x3)
        out = self.up3(out, x2)
        out = self.up4(out, x1)
        return self.outc(out)

refiner = MediSegUNet(3, 1, False).cuda().eval()
_ck = torch.load(CFG.ARTIFACT_ROOT / f"mediseg_refiner_fold{CFG.FOLD}_resume.pt",
                  map_location="cuda", weights_only=False)
refiner.load_state_dict(_ck.get("best_model") or _ck["model"], strict=True)
del _ck

def result_masks(result):
    if result.masks is None:
        return [], np.zeros(0, np.float32)
    h, w = result.orig_shape
    out = []
    for mask in result.masks.data.detach().cpu().numpy():
        if mask.shape != (h, w):
            mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_LINEAR)
        out.append(mask >= 0.5)
    confs = result.boxes.conf.detach().cpu().numpy().astype(np.float32) if result.boxes is not None else np.zeros(0, np.float32)
    return out, confs

def mask_square_box(mask, context=1.65, minimum=96):
    h, w = mask.shape
    ys, xs = np.where(mask)
    if not len(xs):
        return 0, 0, w, h
    cx, cy = (xs.min() + xs.max() + 1) / 2, (ys.min() + ys.max() + 1) / 2
    side = int(math.ceil(max(xs.max() - xs.min() + 1, ys.max() - ys.min() + 1) * context))
    side = min(max(side, minimum), min(h, w))
    x0, y0 = int(round(cx - side / 2)), int(round(cy - side / 2))
    x1, y1 = x0 + side, y0 + side
    if x0 < 0: x1 -= x0; x0 = 0
    if y0 < 0: y1 -= y0; y0 = 0
    if x1 > w: x0 -= x1 - w; x1 = w
    if y1 > h: y0 -= y1 - h; y1 = h
    return max(0, x0), max(0, y0), min(w, x1), min(h, y1)

@torch.no_grad()
def refine_one(image, proposal):
    x0, y0, x1, y1 = mask_square_box(proposal)
    crop = cv2.resize(image[y0:y1, x0:x1], (CFG.CROP_SIZE, CFG.CROP_SIZE), interpolation=cv2.INTER_LINEAR)
    t = torch.from_numpy(crop.transpose(2, 0, 1)[None].copy()).float().cuda() / 255.0
    prob = refiner(t).sigmoid()[0, 0].cpu().numpy()
    prob = cv2.resize(prob, (x1 - x0, y1 - y0), interpolation=cv2.INTER_LINEAR)
    refined = np.zeros_like(proposal, dtype=bool)
    refined[y0:y1, x0:x1] = prob >= CFG.REFINER_THRESHOLD
    return refined & proposal

def greedy_disjoint(masks, order, min_area=CFG.MIN_AREA):
    occupied = np.zeros((2048, 2048), dtype=bool)
    kept = []
    for idx in order:
        ex = masks[idx] & ~occupied
        if ex.sum() < min_area:
            continue
        kept.append(ex); occupied |= ex
    return kept

def drop_contained(masks, thr=CFG.DROP_THRESHOLD):
    if len(masks) < 2:
        return list(range(len(masks)))
    rles = mask_utils.encode(np.asfortranarray(np.stack(masks, -1).astype(np.uint8)))
    areas = np.asarray(mask_utils.area(rles), np.float64)
    si = np.asarray(mask_utils.iou(rles, rles, [0] * len(rles)), np.float64)
    dropped = set()
    for i in range(len(masks)):
        for j in range(i + 1, len(masks)):
            if i in dropped or j in dropped:
                continue
            v = si[i, j]
            if v <= 0:
                continue
            inter = v * (areas[i] + areas[j]) / (1.0 + v)
            s = i if areas[i] <= areas[j] else j
            if inter / max(areas[s], 1.0) >= thr:
                dropped.add(s)
    return [i for i in range(len(masks)) if i not in dropped]

def apply_ownership(masks, scores):
    if not masks:
        return []
    areas = np.array([m.sum() for m in masks], np.float64)
    return greedy_disjoint(masks, sorted(drop_contained(masks), key=lambda i: areas[i]))

def to_rle(ann, h, w):
    seg = ann["segmentation"]
    if isinstance(seg, list):
        return mask_utils.merge(mask_utils.frPyObjects(seg, h, w))
    if isinstance(seg.get("counts"), list):
        return mask_utils.frPyObjects(seg, h, w)
    r = dict(seg)
    if isinstance(r["counts"], str):
        r["counts"] = r["counts"].encode()
    return r

def iou_matrix(truth, pred):
    if not truth or not pred:
        return np.zeros((len(truth), len(pred)))
    return np.asarray(mask_utils.iou(pred, truth, [0] * len(truth))).T

class PanopticQuality:
    def __init__(self, thr=0.5):
        self.thr, self.tp_ious, self.fp, self.fn = thr, [], 0, 0
    def add(self, m, n_gt, n_pred):
        if n_gt == 0:
            self.fp += n_pred; return
        if n_pred == 0:
            self.fn += n_gt; return
        hits = m > self.thr
        self.tp_ious.extend(m[hits].tolist())
        self.fp += int((hits.sum(0) == 0).sum())
        self.fn += int((hits.sum(1) == 0).sum())
    def summary(self):
        tp = len(self.tp_ious)
        den = tp + 0.5 * self.fp + 0.5 * self.fn
        return {"pq": float(sum(self.tp_ious) / den) if den else 0.0,
                "sq": float(sum(self.tp_ious) / tp) if tp else 0.0,
                "rq": float(tp / den) if den else 0.0,
                "tp": tp, "fp": self.fp, "fn": self.fn}

gt_rles = {iid: [to_rle(a, int(info["height"]), int(info["width"]))
                  for a in annotations_by_image[iid] if not a.get("iscrowd", 0)]
           for iid, info in images.items()}

def evaluate(yolo_weight, tag):
    model = YOLO(str(yolo_weight))
    meter = PanopticQuality()
    for i, name in enumerate(tqdm(valid_files_fold, desc=f"{tag} 評価")):
        path = CFG.PREPROC_DIR / name
        res = model.predict(str(path), imgsz=CFG.YOLO_IMGSZ,
                             conf=max(0.08, CFG.SUBMISSION_CONF * 0.75),
                             iou=CFG.PREDICT_IOU, agnostic_nms=True,
                             retina_masks=True, verbose=False)[0]
        masks, confs = result_masks(res)
        props = [(c, m) for c, m in zip(confs, masks) if c >= CFG.SUBMISSION_CONF]
        image = cv2.imread(str(path), cv2.IMREAD_COLOR)
        refined = [refine_one(image, m) for _, m in props]
        preds = apply_ownership(refined, np.ones(len(refined), np.float32))
        prles = []
        if preds:
            enc = mask_utils.encode(np.asfortranarray(np.stack(preds, -1).astype(np.uint8)))
            prles = enc if isinstance(enc, list) else [enc]
        for iid in file_to_ids[name]:
            truth = gt_rles[iid]
            meter.add(iou_matrix(truth, prles), len(truth), len(prles))
        if i % 20 == 0:
            write_status(f"eval:{tag}", i, len(valid_files_fold), name)
    del model
    torch.cuda.empty_cache()
    return meter.summary()

res_prod = evaluate(CFG.RUNS_ROOT / f"yolo_fold_{CFG.FOLD}" / "weights" / "best.pt", "prod")
res_v67 = evaluate(CFG.RUNS_ROOT / f"v67_yolo_fold_{CFG.FOLD}" / "weights" / "best.pt", "v67")

delta = res_v67["pq"] - res_prod["pq"]
manifest = {"fold": CFG.FOLD, "folds_sha256": frozen_fold_sha256, "baseline_recorded": CFG.BASELINE_PQ,
             "prod_rerun": res_prod, "v67": res_v67, "delta_pq": delta,
             "neigh_minutes": CFG.NEIGH_MINUTES, "per_obs": CFG.PER_OBS,
             "verdict": ("adopt-candidate" if delta > 0.005
                           else "noise-reject" if abs(delta) < 0.002 else "reject")}
CFG.MANIFEST.write_text(json.dumps(manifest, indent=1, ensure_ascii=False))
write_status("done")
print(f"prod(再採点): {res_prod}")
print(f"v67(近傍増強): {res_v67}")
print(f"ΔPQ: {delta:+.4f}  -> {manifest['verdict']}")
